# 参照訳なしで翻訳の品質を推定する (品質推定)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_machine_translation/mt_quality_estimation.ipynb)

NLP クックブック 24-16 ／ 機械翻訳

正解の訳 (参照訳) がない状況で、機械翻訳の出力がどれくらい良いかを見積もりたい。

In [1]:
!pip install -q datasets sentence-transformers

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import kendalltau, spearmanr

# WMT 2022 の人手評価 (原文、機械訳、参照訳、人が付けた品質スコア)。英日翻訳の部分を使う
df = load_dataset("RicardoRei/wmt-sqm-human-evaluation", split="train").to_pandas()
df = df[(df["lp"] == "en-ja") & (df["src"].str.len().between(20, 200))].sample(600, random_state=0).reset_index(drop=True)
print(len(df), "件 / システム数", df["system"].nunique(), "/ スコアの分布", df["score"].describe().round(2).to_dict())
for r in df.sort_values("score").iloc[[0, 300, -1]].itertuples():
    print(f"[{r.score:.2f}] {r.src[:60]}\n        → {r.mt[:60]}")

600 件 / システム数 13 / スコアの分布 {'count': 600.0, 'mean': 82.73, 'std': 12.74, 'min': 7.0, '25%': 79.0, '50%': 85.0, '75%': 90.54, 'max': 100.0}
[7.00] Step 6: Return said cold/produce.
        → ステップ6: リターンは言った寒さ/生産します。
[85.00] Go to the Home screen.
        → ホーム画面へ移動します。
[100.00] Many people who hate trans people also hate black people.
        → トランスジェンダーを嫌う人の多くは、黒人も嫌いだ。


In [3]:
from sentence_transformers import SentenceTransformer

# 方法 1: 原文と訳文の、多言語埋め込みによる類似度
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
e_src = embedder.encode(["query: " + s for s in df["src"]], normalize_embeddings=True, batch_size=32)
e_mt = embedder.encode(["query: " + s for s in df["mt"]], normalize_embeddings=True, batch_size=32)
df["埋め込みの類似度"] = (e_src * e_mt).sum(axis=1)

# 方法 2: 長さの比 (極端に短い・長い訳は怪しい、という素朴なベースライン)
ratio = df["mt"].str.len() / df["src"].str.len()
df["長さの比"] = -np.abs(np.log(ratio / ratio.median()))

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 方法 3: LLM に「正確で自然な訳か」を尋ね、「はい」と答える確率をスコアにする
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()
YES_NO = [tokenizer(c, add_special_tokens=False)["input_ids"][0] for c in ["はい", "いいえ"]]

@torch.no_grad()
def p_good(src, mt):
    prompt = ("次の「英語の原文」を日本語に翻訳した「訳文」を評価してください。原文の意味を過不足なく伝えていて、日本語として自然なら「はい」、"
              f"誤訳・訳抜け・不自然な表現があれば「いいえ」と、一語で答えてください。\n\n英語の原文: {src}\n訳文: {mt}")
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return float(llm(ids).logits[0, -1, YES_NO].softmax(-1)[0])

sub = df.head(150).copy()                      # LLM は 1 件あたり数秒かかるので、150 件だけで比べる
sub["LLM (はいの確率)"] = sub["llm"] = [p_good(s, m) for s, m in zip(sub["src"], sub["mt"])]

In [5]:
rows = []
for name in ["長さの比", "埋め込みの類似度", "LLM (はいの確率)"]:
    rows.append({"方法": name, "Spearman": spearmanr(sub["score"], sub[name]).statistic, "Kendall": kendalltau(sub["score"], sub[name]).statistic})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("600 件での埋め込みの類似度:", round(spearmanr(df["score"], df["埋め込みの類似度"]).statistic, 3))

# LLM が低く、人が高く評価した例 / その逆
sub["差"] = sub["LLM (はいの確率)"].rank() - sub["score"].rank()
for r in pd.concat([sub.sort_values("差").head(2), sub.sort_values("差").tail(2)]).itertuples():
    print(f"人 {r.score:.0f} / LLM {r.llm:.2f} | {r.src[:50]} → {r.mt[:50]}")

         方法  Spearman  Kendall
       長さの比     0.081    0.053
   埋め込みの類似度    -0.101   -0.068
LLM (はいの確率)     0.134    0.091
600 件での埋め込みの類似度: -0.103
人 95 / LLM 0.00 | Returns must be within 30 days of delivery in the  → リターンは以内になされなけれ30日の配信とまったく同じ条件でそれらが送信された。
人 100 / LLM 0.00 | You are reading the ebooks on the #PRS_ORG# ereade → あなたは#PRS_ORG#eリーダーで電子書籍を読んでいますよね？
人 76 / LLM 1.00 | Each vintage leather bag is a totally natural and  → ヴィンテージ レザー バッグは、完全に自然で手作りの製品であるため、色や仕上げが 1 点 1 点異な
人 72 / LLM 1.00 | I am saving money to move.” → 私は移動するためにお金を節約しています。”


In [6]:
# はっきり壊れた訳なら見つけられるか: 別の文の訳にすり替えたもの、後半を切り落としたものを混ぜて、元の訳と区別できるかを調べる
from sklearn.metrics import roc_auc_score

good = sub.head(60)
swapped = good["mt"].shift(1, fill_value=good["mt"].iloc[-1]).tolist()          # 別の原文の訳
truncated = [m[:len(m) // 2] for m in good["mt"]]                               # 訳抜け
def embed_score(srcs, mts):
    a = embedder.encode(["query: " + s for s in srcs], normalize_embeddings=True)
    b = embedder.encode(["query: " + s for s in mts], normalize_embeddings=True)
    return (a * b).sum(axis=1)

for label, corrupted in [("別の文の訳にすり替え", swapped), ("後半を切り落とす", truncated)]:
    y = [1] * len(good) + [0] * len(good)
    emb = np.concatenate([embed_score(good["src"], good["mt"]), embed_score(good["src"], corrupted)])
    llm_s = list(good["llm"]) + [p_good(s, m) for s, m in zip(good["src"], corrupted)]
    print(f"{label}: 元の訳と区別できる度合い (AUC) 埋め込み {roc_auc_score(y, emb):.3f} / LLM {roc_auc_score(y, llm_s):.3f}")

別の文の訳にすり替え: 元の訳と区別できる度合い (AUC) 埋め込み 1.000 / LLM 0.980
後半を切り落とす: 元の訳と区別できる度合い (AUC) 埋め込み 0.849 / LLM 0.749


- 品質推定 (quality estimation, QE) は、参照訳を使わずに原文と訳文だけから品質を予測する。翻訳結果をそのまま公開してよいか、人が後編集すべきかを振り分けるのに使われる。
- この評価データは、最近の高品質な翻訳システムの出力で、ほとんどが 80 点以上。その中の細かい優劣を当てるのは非常に難しく、ここで試した 3 つの方法は、どれも人の評価とほとんど相関しなかった。
- 一方、別の文の訳や訳抜けのような、はっきり壊れた訳は区別できる。簡易な方法は「明らかな失敗をはじく」用途に使い、細かい品質の順位付けには使わない。
- 大きな LLM を使った判定は、専用の品質推定モデルに匹敵することが報告されている。小さなモデルで同じ結果は期待できない。
- 専用のモデルとしては、人手評価で学習された CometKiwi が代表的 (利用にはライセンスへの同意が必要)。大量の文を安定して評価するなら、まずこれを検討する。
- 文単位の人手評価はそもそも雑音が大きい。相関係数は 0.3〜0.5 程度でも「使える」とされることが多く、高い値を期待しすぎない。